In [ ]:
%%sql
SELECT
    ah.id AS activity_header_id,
    ah.file_number,
    MAX(
        CASE
            WHEN LOWER(TRIM(ws.description)) IN ('case raised in error', 'bnssg - case raised in error')
            THEN 1
            ELSE 0
        END
    ) AS has_case_raised_in_error,
    CASE
        WHEN MAX(
            CASE
                WHEN LOWER(TRIM(ws.description)) IN ('case raised in error', 'bnssg - case raised in error')
                THEN 1
                ELSE 0
            END
        ) = 1 THEN 0
        ELSE 1
    END AS z_src_is_active
FROM silver_wip_activityheader ah
LEFT JOIN silver_wip_activityentry ae
    ON ae.activity_header_id = ah.id
LEFT JOIN silver_wip_service ws
    ON ae.activity_service_id = ws.id
GROUP BY ah.id, ah.file_number;

In [ ]:
%%sql
SELECT
    ah.id AS activity_header_id,
    ah.file_number,
    ae.activity_service_id,
    ws.description
FROM silver_wip_activityheader ah
LEFT JOIN silver_wip_activityentry ae
    ON ae.activity_header_id = ah.id
LEFT JOIN silver_wip_service ws
    ON ae.activity_service_id = ws.id
WHERE LOWER(TRIM(ws.description)) IN ('case raised in error', 'bnssg - case raised in error');

In [ ]:
%%sql
SELECT
    ae.activity_header_id,
    COUNT(*) AS entry_count
FROM silver_wip_activityentry ae
GROUP BY ae.activity_header_id
HAVING COUNT(*) > 1
ORDER BY entry_count DESC;

In [ ]:
----------------------------------

In [ ]:
,1 as z_record_is_active
,CASE
    WHEN wiperr.has_case_raised_in_error = 1 THEN 0
    ELSE 1
 END AS z_src_is_active

In [ ]:
LEFT JOIN (
    SELECT
        ae.activity_header_id,
        MAX(
            CASE
                WHEN LOWER(TRIM(ws.description)) IN ('case raised in error', 'bnssg - case raised in error')
                THEN 1
                ELSE 0
            END
        ) AS has_case_raised_in_error
    FROM silver_wip_activityentry ae
    LEFT JOIN silver_wip_service ws
        ON ae.activity_service_id = ws.id
    GROUP BY ae.activity_header_id
) wiperr
    ON wiperr.activity_header_id = ah.id

In [ ]:
SELECT
    z_src_system_id,
    z_src_is_active,
    COUNT(*) AS row_count
FROM silver_care_episode
WHERE z_src_system_id = 'WIP'
GROUP BY z_src_system_id, z_src_is_active;

In [ ]:
SELECT TOP 50
    care_epi_id,
    care_epi_src_id,
    z_src_system_id,
    z_src_is_active
FROM silver_care_episode
WHERE z_src_system_id = 'WIP'
  AND z_src_is_active = 0;

In [ ]:
-- Set WIP source active flag based on related activity entry service descriptions

In [ ]:
-- Derive a header-level WIP error flag from activity entry to service mapping to avoid duplicate rows

dev


In [ ]:
full path:-


silver_wip_activityheader.id
→ silver_wip_activityentry.activity_header_id
→ silver_wip_activityentry.activity_service_id
→ silver_wip_service.id
→ silver_wip_service.description


WIP trace summary
The care episode row is based on silver_wip_activityheader.
activityentry links back to the header using activity_header_id, and each entry contains activity_service_id.
The actual service description is stored in silver_wip_service.
The required WIP error values were found in silver_wip_service.description, so the active flag needs to be derived through the header → entry → service path.
Because one header can have many entry rows, the flag has to be derived at header level first to avoid duplicate care episode rows.

In [ ]:
WIP
Implemented CARE EPISODE.Z_SRC_IS_ACTIVE for WIP.

Logic applied
A header-level flag was derived using related activity entry service descriptions.
If any related activity entry maps to a WIP service description of Case raised in error or BNSSG - Case Raised in Error, then z_src_is_active = 0; otherwise z_src_is_active = 1.

Technical note
silver_wip_activityheader is the care episode source grain, while the required service values are held via silver_wip_activityentry -> silver_wip_service.
A direct join would introduce duplicate care episode rows because one activity header can have many activity entries, so an aggregated subquery join was used to derive the flag safely at header level.

Validation
Validated source path and duplicate risk before implementation:

silver_wip_activityentry links to silver_wip_service through activity_service_id
matching values were confirmed in silver_wip_service.description
duplicate risk was confirmed for direct header-to-entry joins
implemented using aggregated join to preserve one row per care episode

In [ ]:
diret join------------------

In [ ]:
%%sql
SELECT
    ah.id AS activity_header_id,
    ah.file_number,
    ae.activity_service_id,
    ws.description,
    CASE
        WHEN LOWER(TRIM(ws.description)) IN ('case raised in error', 'bnssg - case raised in error')
        THEN 0
        ELSE 1
    END AS z_src_is_active_direct
FROM silver_wip_activityheader ah
LEFT JOIN silver_wip_activityentry ae
    ON ae.activity_header_id = ah.id
LEFT JOIN silver_wip_service ws
    ON ae.activity_service_id = ws.id;

In [ ]:
%%sql
--og header count
SELECT COUNT(*) AS header_count
FROM silver_wip_activityheader;

In [ ]:
%%sql
--direct jin count
SELECT COUNT(*) AS direct_join_count
FROM silver_wip_activityheader ah
LEFT JOIN silver_wip_activityentry ae
    ON ae.activity_header_id = ah.id
LEFT JOIN silver_wip_service ws
    ON ae.activity_service_id = ws.id;

In [ ]:
%%sql
--dup proof
SELECT
    ah.id AS activity_header_id,
    ah.file_number,
    COUNT(*) AS row_count_after_direct_join
FROM silver_wip_activityheader ah
LEFT JOIN silver_wip_activityentry ae
    ON ae.activity_header_id = ah.id
LEFT JOIN silver_wip_service ws
    ON ae.activity_service_id = ws.id
GROUP BY ah.id, ah.file_number
HAVING COUNT(*) > 1
ORDER BY row_count_after_direct_join DESC;

In [ ]:
 nnnnn      ];,lm

In [ ]:
%%sql
SELECT DISTINCT
    ae.activity_header_id
FROM silver_wip_activityentry ae
LEFT JOIN silver_wip_service ws
    ON ae.activity_service_id = ws.id
WHERE LOWER(TRIM(ws.description)) IN ('case raised in error', 'bnssg - case raised in error')
ORDER BY ae.activity_header_id;

In [ ]:
%%sql
SELECT
    ae.activity_header_id,
    ae.id AS activity_entry_id,
    ae.activity_service_id
FROM silver_wip_activityentry ae
WHERE ae.activity_header_id = 275
ORDER BY ae.id;

In [ ]:
%%sql
SELECT
    ws.id,
    ws.description
FROM silver_wip_service ws
WHERE ws.id = 310;

In [ ]:
Evidence
Attached screenshots show that activity_header_id = 275 has multiple related rows in silver_wip_activityentry, all mapping to activity_service_id = 310.
In silver_wip_service, id = 310 resolves to Case raised in error.
This confirms the WIP source path and also demonstrates why a direct join would duplicate care episode rows, so the logic was implemented using a header-level aggregated join.

In [ ]:
last

In [ ]:
%%sql
SELECT
    ah.id AS activity_header_id,
    ah.file_number,
    wiperr.has_case_raised_in_error,
    CASE
        WHEN wiperr.has_case_raised_in_error = 1 THEN 0
        ELSE 1
    END AS z_src_is_active_test
FROM silver_wip_activityheader ah
LEFT JOIN (
    SELECT
        ae.activity_header_id,
        MAX(
            CASE
                WHEN LOWER(TRIM(ws.description)) IN ('case raised in error', 'bnssg - case raised in error')
                THEN 1
                ELSE 0
            END
        ) AS has_case_raised_in_error
    FROM silver_wip_activityentry ae
    LEFT JOIN silver_wip_service ws
        ON ae.activity_service_id = ws.id
    GROUP BY ae.activity_header_id
) wiperr
    ON wiperr.activity_header_id = ah.id
WHERE wiperr.has_case_raised_in_error = 1
LIMIT 50;

In [ ]:
%%sql
SELECT
    ah.id AS activity_header_id,
    ah.file_number,
    wiperr.has_case_raised_in_error
FROM silver_wip_activityheader ah
LEFT JOIN (
    SELECT
        ae.activity_header_id,
        MAX(
            CASE
                WHEN LOWER(TRIM(ws.description)) IN ('case raised in error', 'bnssg - case raised in error')
                THEN 1
                ELSE 0
            END
        ) AS has_case_raised_in_error
    FROM silver_wip_activityentry ae
    LEFT JOIN silver_wip_service ws
        ON ae.activity_service_id = ws.id
    GROUP BY ae.activity_header_id
) wiperr
    ON wiperr.activity_header_id = ah.id
WHERE ah.id = 275;

In [ ]:
%%sql
SELECT
    t.care_epi_id,
    t.care_epi_src_id,
    t.z_src_is_active AS test_flag,
    m.z_src_is_active AS main_flag
FROM silver_care_episode_test t
LEFT JOIN silver_care_episode m
    ON t.care_epi_id = m.care_epi_id
WHERE t.z_src_system_id = 'WIP'
  AND t.z_src_is_active = 0
ORDER BY t.care_epi_id;

In [ ]:
%%sql
SELECT
    COUNT(*) AS missing_in_main
FROM silver_care_episode_test t
LEFT JOIN silver_care_episode m
    ON t.care_epi_id = m.care_epi_id
WHERE t.z_src_system_id = 'WIP'
  AND t.z_src_is_active = 0
  AND m.care_epi_id IS NULL;

In [ ]:
WITH mpb_source AS (
    -- MPB source values derived from DRJ appointments and appointment attendances
    -- session_status_src_id includes MPB001 prefix plus the derived source status id

    SELECT DISTINCT
        CASE
            WHEN TRIM(a.cancelledBy) IS NOT NULL AND TRIM(a.cancelledBy) <> ''
                THEN CONCAT(TRIM(att.name), '_', TRIM(a.cancelledBy))
            ELSE TRIM(att.name)
        END AS session_status_src_name,

        CONCAT(
            'MPB001_',
            LOWER(TRIM(
                CASE
                    WHEN TRIM(a.cancelledBy) IS NOT NULL AND TRIM(a.cancelledBy) <> ''
                        THEN CONCAT(TRIM(att.name), '_', TRIM(a.cancelledBy))
                    ELSE TRIM(att.name)
                END
            ))
        ) AS session_status_src_id,

        'MPB001' AS session_status_src_sys_inst_id
    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_appointment_attendances att
        ON a.attendance_id = att.id
    WHERE att.name IS NOT NULL
      AND TRIM(att.name) <> ''
),

wip_source AS (
    -- WIP source values derived from activity entry, activity header and activity status
    -- session_status_src_id includes WIP001 prefix plus the derived source status id

    SELECT DISTINCT
        CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
        END AS session_status_src_name,

        CONCAT(
            'WIP001_',
            LOWER(TRIM(
                CASE
                    WHEN e.is_dna = true THEN 'Did Not Attend'
                    WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
                END
            ))
        ) AS session_status_src_id,

        'WIP001' AS session_status_src_sys_inst_id
    FROM silver_wip_activityentry e
    LEFT JOIN silver_wip_activityheader h
        ON e.activity_header_id = h.id
    LEFT JOIN silver_wip_activitystatus st
        ON h.activity_status_id = st.id
    WHERE CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
          END IS NOT NULL
),

sone_source AS (
    -- S1 / SONE source values derived from SRAppointment and SRMapping
    -- session_status_src_id includes the source instance prefix plus appointment_status

    SELECT DISTINCT
        TRIM(m.mapping) AS session_status_src_name,

        CONCAT(
            'SONE',
            CAST(a.id_organisation_source AS STRING),
            '_',
            CAST(a.appointment_status AS STRING)
        ) AS session_status_src_id,

        'SONE' AS session_status_src_sys_inst_id
    FROM silver_sone_srappointment a
    LEFT JOIN silver_sone_srmapping m
        ON CAST(a.appointment_status AS STRING) = CAST(m.id AS STRING)
    WHERE a.appointment_status IS NOT NULL
      AND a.id_organisation_source IS NOT NULL
      AND m.id IS NOT NULL
      AND m.mapping IS NOT NULL
      AND TRIM(m.mapping) <> ''
)

In [ ]:
WITH mpb_source AS (
    SELECT DISTINCT
        CASE
            WHEN TRIM(a.cancelledBy) IS NOT NULL AND TRIM(a.cancelledBy) <> ''
                THEN CONCAT(TRIM(att.name), '_', TRIM(a.cancelledBy))
            ELSE TRIM(att.name)
        END AS session_status_src_name,
        CONCAT('MPB001_', LOWER(TRIM(CASE WHEN TRIM(a.cancelledBy) IS NOT NULL AND TRIM(a.cancelledBy) <> '' THEN CONCAT(TRIM(att.name), '_', TRIM(a.cancelledBy)) ELSE TRIM(att.name) END))) AS session_status_src_id,
        'MPB001' AS session_status_src_sys_inst_id
    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_appointment_attendances att
        ON a.attendance_id = att.id
    WHERE att.name IS NOT NULL
      AND TRIM(att.name) <> ''
),

wip_source AS (
    SELECT DISTINCT
        CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
        END AS session_status_src_name,
        CONCAT('WIP001_', LOWER(TRIM(CASE WHEN e.is_dna = true THEN 'Did Not Attend' WHEN e.activity_date_time < current_timestamp() THEN 'Attended' END))) AS session_status_src_id,
        'WIP001' AS session_status_src_sys_inst_id
    FROM silver_wip_activityentry e
    LEFT JOIN silver_wip_activityheader h
        ON e.activity_header_id = h.id
    LEFT JOIN silver_wip_activitystatus st
        ON h.activity_status_id = st.id
    WHERE CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
          END IS NOT NULL
),

sone_source AS (
    SELECT DISTINCT
        TRIM(m.mapping) AS session_status_src_name,
        CONCAT('SONE', CAST(a.id_organisation_source AS STRING), '_', CAST(a.appointment_status AS STRING)) AS session_status_src_id,
        'SONE' AS session_status_src_sys_inst_id
    FROM silver_sone_srappointment a
    LEFT JOIN silver_sone_srmapping m
        ON CAST(a.appointment_status AS STRING) = CAST(m.id AS STRING)
    WHERE a.appointment_status IS NOT NULL
      AND a.id_organisation_source IS NOT NULL
      AND m.id IS NOT NULL
      AND m.mapping IS NOT NULL
      AND TRIM(m.mapping) <> ''
)